# Split Datasets

Split dataset to Train/Validation/Test sets

## A. Overview

Random split to Train, Validation and Test sets

In [1]:
%pip install datasketch faiss-cpu sentence-transformers

# # for GPU support, use:
# %pip faiss-cuda

Note: you may need to restart the kernel to use updated packages.


## B. Combine Datasets

In [2]:
from pathlib import Path
import csv
import os
import random
import json
import re
from collections import defaultdict

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup
from src.normalizer import similarity

RANDS_STATE = 42
random.seed(RANDS_STATE)

dataset_path = Path('..') / 'data'

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### B.1. Load Datasets

In [ ]:
# disaster_frac = data_utils.get_data_disaster_fraction()
# disaster_filepath = dataset_path / 'disaster' / str(disaster_frac)

# df_disaster_informative = pd.read_csv(disaster_filepath / 'informative.csv')
# df_disaster_informative['subset'] = 'disaster'
# # df_disaster_humanitarian = pd.read_csv(disaster_filepath / 'humanitarian.csv')
# # df_disaster_humanitarian['subset'] = 'disaster'

In [3]:
extended_filepath = dataset_path / "extended"

df_weather = pd.read_csv(
    extended_filepath / "weather.csv"
)["tweet_text"].to_frame()
df_weather["informative"] = False
df_weather["subset"] = "weather"

# df_out_topic = pd.read_csv(
#     extended_filepath / "out_topic.csv"
# )["tweet_text"].to_frame()
# df_out_topic["informative"] = False
# df_out_topic["subset"] = "out_topic"

### B.2. Combine Datasets

In [4]:
# df_informative = (
#     pd.concat([df_disaster_informative, df_weather, df_out_topic], ignore_index=True)
#     .sample(frac=1, random_state=setup.RANDOM_SEED)
#     .reset_index(drop=True)
# )
# df_informative.head()

## C. Filtering

In [6]:
df_weather_spare = similarity.filter_duplicates_with_resume(df_weather, similarity_threshold=0.75) # 41414

Original dataset size: 47518
Vectorizing text...
Resuming from row 5000. Previously identified 1412 duplicates.
Processed up to row 10000/47518. Identified 2197 near-duplicates in total.
Processed up to row 15000/47518. Identified 2768 near-duplicates in total.
Processed up to row 20000/47518. Identified 3734 near-duplicates in total.
Processed up to row 25000/47518. Identified 4317 near-duplicates in total.
Processed up to row 30000/47518. Identified 4777 near-duplicates in total.
Processed up to row 35000/47518. Identified 5419 near-duplicates in total.
Processed up to row 40000/47518. Identified 5971 near-duplicates in total.
Processed up to row 45000/47518. Identified 6078 near-duplicates in total.
Processed up to row 47518/47518. Identified 6104 near-duplicates in total.
Processing complete. Checkpoint file removed.
Final dataset size after near-duplicate removal: 41414


In [ ]:
df_weather_faiss = similarity.filter_duplicates_faiss(df_weather) # 41414

Original dataset size: 47518
Loading embedding model...
Vectorizing text to dense representations...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
# # The df_out_topic is too large
# # split to multiple partitions
# # accept similar records accross them
# partition_records = 1_000_000

# df_out_topic_processed = None

# for i in range(0, len(df_out_topic) // partition_records):
#     start_idx = i * partition_records
#     end_idx = min((i + 1) * partition_records, len(df_out_topic))
#     partition = df_out_topic.iloc[start_idx:end_idx]
#     partition_processed = similarity.filter_duplicates_with_resume(partition)
#     # if i == 0:
#     #     df_out_topic_processed = partition_processed
#     # else:
#     df_out_topic_processed = pd.concat([df_out_topic_processed, partition_processed], ignore_index=True)

Original dataset size: 1000000
Vectorizing text...
Starting fresh chunked cosine similarity...


In [ ]:
# df_out_topic_processed = similarity.filter_duplicates_with_resume(df_out_topic) # 41414

Original dataset size: 10403525
Running exact duplicate pre-pass...
Removed 21260 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...
Final dataset size after near-duplicate removal: 10233915


In [ ]:
# df_out_topic_processed.to_csv(extended_filepath / "out_topic_deduplicated.csv", index=False, quoting=csv.QUOTE_ALL)